# Chapter 0 — Setup and foundations

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


# PART 0 — GETTING SET UP

You can read this book without running anything. You will not learn it that way. Every person who reports getting real value from this course reports typing the code and breaking it.

## 0.1 What you need

A computer with at least 8 GB of RAM. Chapters 1 through 6 and 8 run on any laptop, including a several-year-old one, with no special hardware. Chapter 7 will run on a laptop but slowly; the full result needs a GPU. Chapter 9 needs several GPUs or about $10 of rented cloud time.

**What a GPU is.** A **CPU** (central processing unit) is your computer's general-purpose brain: a handful of very fast, very flexible cores. A **GPU** (graphics processing unit) is a slab of thousands of simple cores that all do the same arithmetic at the same time on different data. Neural networks are almost entirely "multiply these thousands of numbers by those thousands of numbers," which is exactly the shape a GPU is built for. That is why training on a GPU can be 10 to 100 times faster than on a CPU. [standard]

## 0.2 Installing Python and the libraries

**Step 1: get Python.** Download Python 3.10 or newer from [python.org](https://www.python.org/downloads/), or on a Mac use [Homebrew](https://brew.sh) (`brew install python`). Check it worked by opening a terminal and typing:

**bash:**

```bash
python3 --version
```

You should see something like `Python 3.12.3`. (That is the version this book was written on. [verified])

**Step 2: make a virtual environment.** A **virtual environment** is a private folder holding this project's libraries, so that installing something here cannot break another project. Standard practice, worth the 20 seconds.

**bash:**

```bash
mkdir zero-to-hero && cd zero-to-hero
python3 -m venv .venv
source .venv/bin/activate      # on Windows: .venv\Scripts\activate
```

Your prompt now shows `(.venv)`, meaning that environment is active. You repeat only the `source` line in future sessions.

**Step 3: install the libraries.**

**bash:**

```bash
pip install torch numpy matplotlib jupyter
```

- **torch** is PyTorch, the library that provides fast arrays and automatic derivatives.
- **numpy** handles numeric arrays; PyTorch borrows its conventions.
- **matplotlib** draws the plots you will use to diagnose training.
- **jupyter** is the notebook interface.

This downloads a few hundred megabytes and takes a couple of minutes.

**Step 4: start the notebook.**

**bash:**

```bash
jupyter notebook
```

A browser tab opens. Click **New → Python 3** to create a notebook. You now have a page of empty **cells**. Type code into one and press **Shift+Enter** to run it. The output appears directly underneath, and anything you defined stays in memory for the next cell.

**Run it.**

In [ ]:
import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())

**What you should see** (your version will differ; `False` for the GPU is fine for Chapters 1–6 and 8):

**Expected output:**

```
torch 2.11.0+cu130 | GPU available: True
```

[verified]

## 0.3 Getting the two datasets

The whole course uses exactly two files.

**bash:**

```bash
curl -O https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
curl -O https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
```

**Run it.**

In [ ]:
words = open('names.txt', 'r').read().splitlines()
print(len(words), "names |", words[:5])
text = open('input.txt', 'r').read()
print(len(text), "characters of Shakespeare")

**What you should see:**

**Expected output:**

```
32033 names | ['emma', 'olivia', 'ava', 'isabella', 'sophia']
1115394 characters of Shakespeare
```

[verified]

Those two numbers, 32,033 names and about 1.1 million characters, are the ones quoted throughout the lectures as "32,000 names" and "1 megabyte of Shakespeare."

## 0.4 How to work through a chapter

1. Read the chapter's "The problem" section before any code.
2. Type each code cell rather than pasting it. Typos teach you the API.
3. Before running a cell, say out loud what you expect it to print. Being wrong is the useful part.
4. When a shape error appears, print `.shape` on everything involved. Most errors in this field are shape errors.
5. At the end of a chapter, close the notebook and rebuild the key piece from a blank file.

## 0.5 If something goes wrong

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'torch'` | Virtual environment not active, or Jupyter is using a different Python | Re-run `source .venv/bin/activate`, then `pip install torch`, then restart Jupyter |
| `FileNotFoundError: names.txt` | The notebook's working folder differs from where you downloaded | Run `import os; print(os.getcwd())` and move the file there |
| Numbers differ slightly from this book | Different random seed or hardware | Expected. Set `torch.manual_seed(1337)` to match closely; last-digit differences remain and are harmless |
| Cell hangs forever | A training loop with too many steps | Press **I, I** (interrupt twice) or the stop button; lower the step count |
| `CUDA out of memory` | Model or batch too large for the GPU | Lower `batch_size`, then `block_size` |

---

# PART I — FOUNDATIONS

*Everything needed before Chapter 1. Each concept has a concrete example, and anything a concept leans on is broken down first. If you already know this material, skim the bold text and move on.*

## 1.1 What a model is

A **model** is a function: numbers go in, numbers come out.

Predicting a house price from its size, you might guess `price = 200 × square_feet`. Feed in 1,000, get 200,000. That rule is a model, and the `200` is a **parameter**: a number you are free to change to fit reality better.

**Training** means finding parameter values that make the outputs match reality on examples you already have.

> **Say it to a six-year-old.** Imagine a machine with a lot of little knobs on the side. You put a picture of a cat in the front, and a word comes out the back. At first it says "dog," which is wrong. So you turn the knobs a tiny bit, and try again. If you do that a million times, eventually it says "cat" every time. Nobody tells the machine what a cat is. It just turns knobs until it stops being wrong.

> **For the PhD in the room.** Formally we are choosing a function from a parameterized family, f(x; θ) with θ ∈ ℝⁿ, by minimizing an empirical risk, that is the average of a loss ℓ over a finite sample drawn from an unknown data distribution. Everything interesting hides in the gap between that empirical average and the true expected risk, which is the subject of generalization theory. This course is deliberately empirical about that gap: it measures it with a held-out split rather than bounding it.

## 1.2 The math notation used in this book

Only a handful of symbols appear anywhere in the course. Here they all are.

| Symbol | Read it as | Meaning | Example |
|---|---|---|---|
| Σ | "sum of" | Add up a list | Σ of [1,2,3] is 6 |
| Π | "product of" | Multiply a list together | Π of [1,2,3] is 6 |
| `x²` | "x squared" | x times itself | 3² = 9 |
| `√x` | "square root of x" | The number that squares to x | √9 = 3 |
| `e` | "Euler's number" | The constant 2.71828... | see 1.4 |
| `exp(x)` or `eˣ` | "e to the x" | e multiplied by itself x times | exp(1) = 2.718 |
| `log(x)` | "natural log of x" | The power you raise e to, to get x | log(2.718) = 1 |
| `∂y/∂x` | "the derivative of y with respect to x" | How much y moves when x is nudged | see 1.5 |
| `∇` | "gradient" | All the derivatives at once, one per parameter | see 1.6 |
| `θ` | "theta" | Standard letter for "all the parameters" | |
| `ℝⁿ` | "R to the n" | The space of lists of n real numbers | ℝ³ is all 3-number lists |

That is the whole vocabulary. Anything else gets defined where it appears.

## 1.3 Numbers, vectors, matrices, tensors

- A **scalar** is one number: `4.2`.
- A **vector** is an ordered list: `[0.3, -1.2, 4.0]`. Picture an arrow in space, or three dials.
- A **matrix** is a grid, rows by columns. A 27×27 matrix holds 729 numbers. You will build exactly that in Chapter 2.
- A **tensor** is the general term: a block of numbers with any number of dimensions. Scalar = 0-D, vector = 1-D, matrix = 2-D, stack of matrices = 3-D. [standard]

**Shape** is the list of sizes per dimension. Shape `32 × 3 × 10` means 32 examples, each 3 things, each described by 10 numbers. That exact shape appears in Chapter 3. [transcript]

**Run it.**

In [ ]:
import torch
a = torch.tensor(4.2)                      # scalar
b = torch.tensor([0.3, -1.2, 4.0])         # vector
c = torch.zeros(27, 27)                    # matrix
d = torch.randn(32, 3, 10)                 # 3-D tensor of random numbers
for name, t in [('a', a), ('b', b), ('c', c), ('d', d)]:
    print(f"{name}: shape {tuple(t.shape)}  dims {t.dim()}  total numbers {t.numel()}")

**What you should see:**

**Expected output:**

```
a: shape ()  dims 0  total numbers 1
b: shape (3,)  dims 1  total numbers 3
c: shape (27, 27)  dims 2  total numbers 729
d: shape (32, 3, 10)  dims 3  total numbers 960
```

[verified]

**Why tensors instead of loops.** A GPU can multiply a million pairs of numbers in roughly the time a Python loop takes to do a few thousand. Every performance idea in this course is some version of "rearrange this into one big tensor operation."

> **Say it to a six-year-old.** A number is one Lego brick. A vector is a row of bricks. A matrix is a flat square of bricks. A tensor is any Lego shape at all, including a big cube. The computer likes it when you hand it the whole cube at once instead of one brick at a time.

## 1.4 Exponentials and logarithms

These two show up constantly, so here they are properly.

**Exponential.** `exp(x)` means the constant e (2.71828...) raised to the power x. Two properties are all that matter:

1. **It is always positive.** `exp(−100)` is a tiny positive number, never negative. This is why it is used to turn arbitrary network outputs into probabilities, which must be positive.
2. **It grows fast and it stretches differences.** `exp(2) ≈ 7.4` and `exp(4) ≈ 54.6`. A gap of 2 in the input became a factor of 7 in the output.

**Logarithm.** `log(x)` is the inverse: the power you must raise e to in order to get x. `log(1) = 0`, `log(2.718) = 1`, `log(0.5) ≈ −0.69`. Three properties matter:

1. **The log of a number between 0 and 1 is negative**, and it plunges toward negative infinity as the number approaches zero. `log(0.1) ≈ −2.3`, `log(0.001) ≈ −6.9`, `log(0)` is undefined, reported as `-inf`.
2. **Logs turn multiplication into addition**: `log(a×b) = log(a) + log(b)`. This is why probabilities, which get multiplied together and quickly become unimaginably small, are handled in log space instead.
3. **It is monotonic**: bigger input, bigger output. So maximizing a probability and maximizing its log are the same problem, and the log version is numerically safer.

**Run it.**

In [ ]:
import math
for x in [0.001, 0.1, 0.5, 1.0, 2.718281828, 10.0]:
    print(f"x={x:<12} exp(x)={math.exp(x):<20.6f} log(x)={math.log(x):.6f}")
print("\nlog turns multiplication into addition:")
print("  log(0.2 * 0.3) =", math.log(0.2*0.3))
print("  log(0.2) + log(0.3) =", math.log(0.2) + math.log(0.3))

**What you should see:**

**Expected output:**

```
x=0.001        exp(x)=1.001001             log(x)=-6.907755
x=0.1          exp(x)=1.105171             log(x)=-2.302585
x=0.5          exp(x)=1.648721             log(x)=-0.693147
x=1.0          exp(x)=2.718282             log(x)=0.000000
x=2.718281828  exp(x)=15.154262            log(x)=1.000000
x=10.0         exp(x)=22026.465795         log(x)=2.302585

log turns multiplication into addition:
  log(0.2 * 0.3) = -2.8134107167600364
  log(0.2) + log(0.3) = -2.8134107167600364
```

[verified]

> **Say it to a six-year-old.** Some numbers are so tiny that writing them down takes forever, like a decimal point with twenty zeros after it. A logarithm is a shortcut for saying "how many zeros," so you can talk about tiny things with small, easy numbers.

## 1.5 A derivative is a sensitivity measurement

This is the one piece of calculus the course requires, and you can get it entirely from nudging.

Take `f(x) = 3x² − 4x + 5`. (Karpathy opens Lecture 1 with this exact function. [transcript]) At `x = 3` it equals 20.

Nudge the input by a tiny `h = 0.001` and recompute: `f(3.001) = 20.014003`. The output moved 0.014003 when the input moved 0.001. Divide: `0.014003 / 0.001 = 14.003`.

That number, essentially 14, is the **derivative** at `x = 3`. It answers exactly one question: *if I wiggle this input slightly, how much and in which direction does the output move?*

- Positive: pushing input up pushes output up.
- Negative: pushing input up pulls output down.
- Near zero: the output does not currently care about this input.

**Run it.**

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

h = 0.001
for x in [3.0, -3.0, 2/3]:
    slope = (f(x+h) - f(x)) / h
    print(f"at x={x:+.4f}  f(x)={f(x):8.4f}  slope≈{slope:+.4f}")

**What you should see:**

**Expected output:**

```
at x=+3.0000  f(x)= 20.0000  slope≈+14.0030
at x=-3.0000  f(x)= 44.0000  slope≈-21.9970
at x=+0.6667  f(x)=  3.6667  slope≈+0.0030
```

[verified]

Read those three lines carefully, because they are the whole idea. At x=3 the function climbs steeply. At x=−3 it falls steeply. At x=2/3 the slope is essentially zero, which means we are sitting at the bottom of the curve. **Finding where the slope is zero is finding the minimum**, and minimizing is what training does.

**Why h = 0.001 and not smaller.** In theory the derivative is the limit as h approaches 0. In practice, computers store numbers with finite precision, so an h that is too small makes `f(x+h)` and `f(x)` round to the same value and the answer collapses to garbage. Try `h = 1e-15` and watch it break. This is one reason real systems compute derivatives symbolically, with rules, rather than numerically, with nudges.

> **Say it to a six-year-old.** Stand on a hill and shuffle one small step forward. Did you go up or down, and by how much? That is all a derivative is: a way of asking "which way is downhill from right here."

> **For the PhD in the room.** What is described here is a forward finite difference, with error O(h) plus floating-point cancellation error O(ε/h), minimized around h ≈ √ε ≈ 1.5e-8 for float64. The course uses it only as a gradient check. Everything real uses reverse-mode automatic differentiation, which is exact up to floating-point and costs one backward sweep for all partial derivatives simultaneously, versus n forward evaluations for finite differences. That asymmetry is why reverse mode won: for a scalar loss and n parameters, reverse mode is O(1) sweeps and forward mode is O(n).

## 1.6 The gradient

When a function has many inputs, each input gets its own sensitivity number. The whole collection is the **gradient**: one number per parameter, each saying "here is how the output moves when you nudge *this* knob, holding the others still."

A network with 41 parameters has a gradient of 41 numbers. GPT-3, with 175 billion parameters, has a gradient of 175 billion numbers, recomputed at every training step. [transcript]

> **Say it to a six-year-old.** You have a hundred knobs and one score. The gradient is a list that says, for every single knob, "turning this one up makes the score a little better" or "a little worse." Then you turn all hundred at once, each in the good direction.

## 1.7 The chain rule

Turning a crank turns a gear, and the gear moves a belt.

- One turn of the crank turns the gear 3 times.
- One turn of the gear moves the belt 2 centimeters.

How far does the belt move per crank turn? `3 × 2 = 6` centimeters.

That multiplication is the **chain rule**: when effects pass through a chain of steps, sensitivities multiply along the chain. [standard]

This is the entire mathematical content of backpropagation. A network is a long chain of small operations, and "how sensitive is the final error to this one weight buried deep inside" is answered by multiplying local sensitivities along the path from that weight to the output. Everything else is bookkeeping to do it for millions of paths at once without losing track.

**Two rules cover almost everything you will meet:**

| Operation | Forward | Backward |
|---|---|---|
| Addition, `c = a + b` | Add the inputs | Pass the incoming sensitivity to both inputs unchanged |
| Multiplication, `c = a × b` | Multiply the inputs | Give each input the *other* input's value, times the incoming sensitivity |

Why addition passes through unchanged: if `c = a + b`, nudging `a` by 0.001 moves `c` by exactly 0.001, a sensitivity of 1, and multiplying by 1 changes nothing.

Why multiplication swaps: if `c = a × b` with `a = 2, b = −3`, then nudging `a` up by 1 changes `c` by `b`, which is −3. So `a`'s sensitivity is `b`'s value, and vice versa.

**Run it.** Verify the swap by nudging, before trusting any formula:

In [ ]:
a, b, h = 2.0, -3.0, 0.0001
c = a * b
print("d(c)/d(a) numerically:", ((a+h)*b - a*b) / h, " ... which is b =", b)
print("d(c)/d(b) numerically:", (a*(b+h) - a*b) / h, " ... which is a =", a)

**What you should see:**

**Expected output:**

```
d(c)/d(a) numerically: -3.000000000010772  ... which is b = -3.0
d(c)/d(b) numerically: 2.0000000000042206  ... which is a = 2.0
```

[verified]

Those trailing digits are floating-point noise from subtracting two nearly equal numbers, which is the effect described in 1.5.

> **Say it to a six-year-old.** If you get two stickers for every drawing, and two candies for every sticker, then each drawing is worth four candies. You just multiply along the chain.

> **For the PhD in the room.** For f: ℝⁿ → ℝᵐ composed with g, the chain rule is the Jacobian product J_(g∘f) = J_g · J_f. Reverse-mode AD never materializes those Jacobians; it evaluates vector-Jacobian products vᵀJ, which for a scalar loss means starting with v = 1 and sweeping backward. Each primitive supplies a VJP rather than a full Jacobian, which is why an elementwise operation over a million values costs a million multiplications rather than a million-squared matrix.

## 1.8 Matrix multiplication, done by hand once

Nearly all of a neural network's arithmetic is matrix multiplication, so do one manually and you will never be confused by it again.

To multiply matrix **A** (2 rows, 3 columns) by matrix **B** (3 rows, 2 columns), you take each row of A, pair it with each column of B, multiply element by element, and sum.

**Expected output:**

```
A = [1 2 3]      B = [ 7  8]
    [4 5 6]          [ 9 10]
                     [11 12]
```

Result entry at row 1, column 1: `(1×7) + (2×9) + (3×11) = 7 + 18 + 33 = 58`.
Row 1, column 2: `(1×8) + (2×10) + (3×12) = 8 + 20 + 36 = 64`.
Row 2, column 1: `(4×7) + (5×9) + (6×11) = 28 + 45 + 66 = 139`.
Row 2, column 2: `(4×8) + (5×10) + (6×12) = 32 + 50 + 72 = 154`.

**The shape rule.** `(2×3) @ (3×2)` gives `(2×2)`. The inner numbers must match, and they vanish; the outer numbers survive. If the inner numbers do not match, the operation is undefined and PyTorch raises an error. **Roughly 90% of the errors you will hit in this course are this rule being violated** [my read], so it is worth memorizing in this form: **the inner dimensions must agree**.

**Run it.**

In [ ]:
import torch
A = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
B = torch.tensor([[7., 8.], [9., 10.], [11., 12.]])
print("A", tuple(A.shape), "@ B", tuple(B.shape), "->", tuple((A @ B).shape))
print(A @ B)

**What you should see:**

**Expected output:**

```
A (2, 3) @ B (3, 2) -> (2, 2)
tensor([[ 58.,  64.],
        [139., 154.]])
```

[verified]

**Why this operation, of all operations.** A neural network layer computes "every output is a weighted sum of every input." That is precisely a matrix multiply: the weight matrix holds one column of weights per output neuron, and multiplying by it computes all the weighted sums simultaneously.

> **Say it to a six-year-old.** You have three ingredients and two recipes. Each recipe says how much of each ingredient to use. Matrix multiplication is working out how much of each ingredient you need for all the recipes at once, in one go, instead of doing each recipe separately.

## 1.9 Broadcasting, and why it silently ruins models

**Broadcasting** is PyTorch automatically stretching a smaller tensor to match a bigger one so an operation can proceed. Add a 3-element vector to a 2×3 matrix, and the vector is copied onto both rows. [standard]

The rules, compared right to left across the shapes:

1. Dimensions match, or
2. one of them is 1 (that one gets stretched), or
3. one of them does not exist (treated as 1).

Otherwise it errors.

This is convenient and it is the single most dangerous feature in the library, because a shape mistake often does not raise an error. It produces a *different, valid, wrong* computation.

**Run it.** The classic bug, normalizing rows versus columns. Note the matrix is **square**, which is what makes the bug silent, and the real counts matrix in Chapter 2 is 27×27:

In [ ]:
import torch
N = torch.tensor([[1., 1., 2.], [3., 3., 6.], [2., 4., 4.]])
right = N / N.sum(1, keepdim=True)   # shape (3,1) -> stretched across columns: CORRECT
wrong = N / N.sum(1)                 # shape (3,)  -> stretched across ROWS: WRONG, no error
print("row sums with keepdim:", tuple(N.sum(1, keepdim=True).shape))
print("row sums without:     ", tuple(N.sum(1).shape))
print("correct rows sum to:", right.sum(1))
print("wrong   rows sum to:", wrong.sum(1))

**What you should see:**

**Expected output:**

```
row sums with keepdim: (3, 1)
row sums without:      (3,)
correct rows sum to: tensor([1., 1., 1.])
wrong   rows sum to: tensor([0.5333, 1.6000, 1.2333])
```

[verified]

The second version ran happily and produced rows that do not sum to 1, meaning they are not probabilities, meaning the model is quietly broken. **The defensive habit: always pass `keepdim=True` when you sum for normalization, and always check that your probabilities sum to 1.**

**Worth trying yourself:** change the matrix to non-square, say 2×3, and run the same two lines. The wrong version now raises `RuntimeError: The size of tensor a (3) must match the size of tensor b (2)`. [verified] A rectangular shape catches the mistake for you; a square shape does not. This is why the bug is so common in the bigram model, where the matrix is 27×27.

> **Say it to a six-year-old.** If one kid brings a bag of sweets to share with a row of friends, everyone gets some. Broadcasting is the computer sharing one small list out across a big table of numbers. It is helpful, but if it shares along the wrong direction, everyone gets the wrong thing and nobody complains.

## 1.10 What a neuron is

An artificial **neuron** does three things: [standard]

1. **Weighted sum.** Multiply each input by its own weight and add them. Inputs `[2.0, 3.0]`, weights `[−3.0, 1.0]`: `2.0×(−3.0) + 3.0×1.0 = −3.0`.
2. **Add a bias**, a per-neuron offset applied regardless of input. Bias `6.5` gives `3.5`. The bias sets how easily the neuron activates at all.
3. **Squash through a nonlinearity.** The course uses `tanh`, which maps any number into the range −1 to 1. `tanh(3.5) ≈ 0.998`.

**Why squash?** Without a bend somewhere, stacking layers is algebraically pointless: a chain of pure multiply-and-add collapses into a single multiply-and-add. The nonlinearity is what makes depth buy you anything.

**Run it.** Prove the collapse to yourself:

In [ ]:
import torch
torch.manual_seed(1337)
x = torch.randn(1, 4)
W1, W2 = torch.randn(4, 5), torch.randn(5, 3)
two_linear_layers = (x @ W1) @ W2
one_equivalent_layer = x @ (W1 @ W2)
print("difference:", (two_linear_layers - one_equivalent_layer).abs().max().item())

**What you should see:**

**Expected output:**

```
difference: 4.76837158203125e-07
```

[verified]

Zero, up to floating-point noise (that is 0.00000048, the rounding error of 32-bit arithmetic, not a real difference). Two stacked linear layers are exactly one linear layer, so without nonlinearities a 50-layer network has the expressive power of a 1-layer network.

**What tanh looks like.** It passes through zero at zero, rises steeply near the middle, and flattens out toward ±1 at the edges. Those flat edges matter enormously and are the subject of Chapter 4: where the curve is flat, the derivative is near zero, and a neuron sitting out there stops learning.

**Run it.**

In [ ]:
import torch
for v in [-4.0, -1.0, 0.0, 1.0, 4.0]:
    t = torch.tensor(v)
    print(f"tanh({v:+.1f}) = {torch.tanh(t):+.6f}   local slope = {1 - torch.tanh(t)**2:.6f}")

**What you should see:**

**Expected output:**

```
tanh(-4.0) = -0.999329   local slope = 0.001341
tanh(-1.0) = -0.761594   local slope = 0.419974
tanh(+0.0) = +0.000000   local slope = 1.000000
tanh(+1.0) = +0.761594   local slope = 0.419974
tanh(+4.0) = +0.999329   local slope = 0.001341
```

[verified]

Look at the slope column. At the edges it is 0.0013, effectively zero. A neuron pushed out there passes almost no gradient backward, so it barely trains. That single fact explains a large fraction of Chapter 4.

> **Say it to a six-year-old.** A neuron is a tiny voter. It listens to a few friends, trusts some of them more than others, has its own mood about the whole thing, and then says yes or no. A brain-sized pile of these tiny voters, all shouting at each other, is what a neural network is.

## 1.11 Layers and networks

A **layer** is a row of neurons all looking at the same inputs. A **multilayer perceptron (MLP)** is several layers stacked, each feeding the next.

- Lecture 1's MLP: **41 parameters** [verified] [transcript].
- Chapter 3's: about 3,400, then about 12,000.
- Chapter 7's transformer: about 10 million.
- GPT-3: 175 billion. [transcript]

The structure is the same at every scale. Only the count changes, which is the most important sentence in this book. [my read]

## 1.12 Probability, softmax, and cross-entropy

From Chapter 2 onward, models do not output "the answer." They output a **probability distribution**: one number per possible outcome, all positive, summing to 1.

For the letter after `a`, a model might say `n` 18%, `r` 12%, `l` 9%, and so on across 27 options.

**How raw outputs become probabilities.** The network emits arbitrary numbers called **logits**, possibly negative, possibly huge. Two steps fix that:

1. **Exponentiate** each (`eˣ`), making everything positive and stretching differences (section 1.4).
2. **Divide by the total**, so they sum to 1.

That recipe is **softmax**.

**Run it.**

In [ ]:
import torch
logits = torch.tensor([2.0, 1.0, 0.1, -5.0])
counts = logits.exp()
probs = counts / counts.sum()
print("logits: ", logits.tolist())
print("exp:    ", [round(c, 4) for c in counts.tolist()])
print("probs:  ", [round(p, 4) for p in probs.tolist()])
print("sums to:", probs.sum().item())
print("matches torch.softmax:", torch.allclose(probs, torch.softmax(logits, dim=0)))

**What you should see:**

**Expected output:**

```
logits:  [2.0, 1.0, 0.10000000149011612, -5.0]
exp:     [7.3891, 2.7183, 1.1052, 0.0067]
probs:   [0.6586, 0.2423, 0.0985, 0.0006]
sums to: 1.0
matches torch.softmax: True
```

[verified]

Note what softmax did: a logit gap of 1.0 (from 2.0 down to 1.0) became a probability ratio of about 2.7×, and the logit of −5 was crushed to 0.06%. Softmax is aggressive about differences. Notice also that `0.1` printed as `0.10000000149011612`: 32-bit floating point cannot represent 0.1 exactly. This is normal and is why you compare floats with `torch.allclose` rather than `==`.

**Scoring a probabilistic model.** If the truth was `n` and the model said 18%, was that good? The measurement used everywhere here is **negative log likelihood**:

1. Take the probability assigned to what actually happened: 0.18.
2. Take its log: `log(0.18) ≈ −1.715`. Negative, and plunging as probability approaches zero.
3. Flip the sign, so confident-and-right scores low: `1.715`.
4. Average over every prediction in the dataset.

That average is the **loss**, also called **cross-entropy loss**. Lower is better; zero is perfect and unreachable in practice.

**Run it.**

In [ ]:
import torch, torch.nn.functional as F
logits = torch.tensor([[2.0, 1.0, 0.1, -5.0]])
target = torch.tensor([0])                     # the correct class is index 0
p = torch.softmax(logits, dim=1)[0, 0]
print("probability given to the truth:", p.item())
print("negative log likelihood by hand:", -torch.log(p).item())
print("F.cross_entropy:               ", F.cross_entropy(logits, target).item())

**What you should see:**

**Expected output:**

```
probability given to the truth: 0.6586053967475891
negative log likelihood by hand: 0.41763070225715637
F.cross_entropy:                0.41763073205947876
```

[verified]

The two numbers agree to seven decimal places and differ in the eighth. That difference is not a bug in either one: they perform the same arithmetic in a different order, and floating point is not perfectly associative. Get used to seeing this; it is why "the loss changed by 1e-8" never means anything.

`F.cross_entropy` is softmax and negative-log-likelihood fused into one function. Use the built-in rather than writing the two steps yourself: it is faster, and it handles the case where a logit is so large that `exp()` overflows to infinity. (It subtracts the maximum logit first, which changes nothing mathematically because softmax is invariant to a constant shift, and everything numerically.)

**Reading loss numbers, the most useful skill in this book.** Before training anything, compute what the loss should be if the model knows nothing. With 27 equally likely characters, the true one gets probability 1/27, so the loss is `−log(1/27) = 3.2958`.

In [ ]:
import math
for vocab in [27, 65, 50257]:
    print(f"vocab {vocab:>6}: a know-nothing model should start at loss {math.log(vocab):.4f}")

**Expected output:**

```
vocab     27: a know-nothing model should start at loss 3.2958
vocab     65: a know-nothing model should start at loss 4.1744
vocab  50257: a know-nothing model should start at loss 10.8249
```

[verified]

Those three numbers appear throughout the rest of the book. In Chapter 4 a network starts at **27.88** instead of 3.2958, which is how you know it is broken before wasting an hour training it. [verified]

> **Say it to a six-year-old.** The machine has to guess which letter comes next, and it is allowed to say "I'm 70% sure it's an A, 20% sure it's a B." If the answer turns out to be A, it gets a small penalty because it was mostly right. If it had said "1% sure it's an A," it gets a huge penalty. The game is to make the total penalty as small as possible, so it learns to be confident only when it should be.

> **For the PhD in the room.** Cross-entropy H(p, q) = −Σ p(x) log q(x) with p the empirical one-hot distribution reduces to negative log likelihood, so minimizing it is maximum likelihood estimation. The gap between it and the entropy of the data is the KL divergence, which is why the loss has a nonzero floor set by the true conditional entropy of English. Character-level English is usually quoted around 1.0 to 1.3 bits per character, roughly 0.7 to 0.9 nats, so the Shakespeare model's 1.48 nats in Chapter 7 is still well above the information-theoretic floor. Also worth flagging: perplexity, common in the language modeling literature, is just exp(loss), so 1.48 nats is a perplexity of about 4.4.

## 1.13 What a language model is

A **language model** assigns probabilities to what comes next in a sequence. [standard]

**Character-level**, used for most of the course: given `emm`, predict the next character. **Sub-word level**, used by ChatGPT: given `The capital of France is`, predict the next chunk of text.

To **generate**, you sample: ask for a distribution over next characters, draw one at random according to those probabilities, append it, ask again. Repeat until an end marker. This is **autoregressive** generation: the model's own output becomes its next input. [standard]

**Why sample randomly instead of always taking the most likely character?** Because always taking the maximum produces repetitive, degenerate text; the model would emit the same name every time. Randomness in proportion to the model's confidence is what makes generation produce variety.

## 1.14 Train, validation, test

Split your data three ways, typically 80% / 10% / 10%: [transcript]

- **Training set** — used to set the parameters.
- **Validation set** (also called dev) — used to choose settings like layer size and learning rate.
- **Test set** — used rarely, ideally once, for an honest final number.

**Why three and not two.** Every time you look at a set and change something in response, you leak a little of that set into your model. The test set stays sealed so the final number means something.

**Overfitting** is when the model memorizes the training data instead of learning the pattern: training loss keeps dropping while validation loss rises. **Underfitting** is when both are high and close together, meaning the model is too small or undertrained.

**Analogy.** Training on past exam papers is fine. Memorizing the answer key is overfitting: perfect on those papers, useless on the real exam. The validation set is a mock exam, and the test set is the real one.

> **For the PhD in the room.** The three-way split is doing model selection and evaluation with the same finite sample, so the validation estimate is optimistically biased by the number of configurations you compare against it, which is a multiple-comparisons problem. Nested cross-validation is the principled fix and is rarely used at this scale because a single training run is expensive. Note too that the split here is by *word*, not by example, which matters: splitting by example would leak, since two examples from the same name share context.

---

# PART II — THE GAP

## 2.1 The problem this course solves

Modern libraries let you train a neural network in about ten lines. Import, choose a model, call `.fit()`. This works, and teaches you almost nothing.

The result is a large population of practitioners who can operate the machinery but cannot diagnose it. When training silently fails, when the loss plateaus at a suspicious number, when the model works on training data and collapses on new data, ten-lines fluency runs out. The knowledge needed is inside the abstraction, and the abstraction exists to hide it.

Karpathy's own version of the complaint, from Lecture 1: he went looking for how `tanh` is actually implemented in PyTorch's source, "spent about 15 minutes and I couldn't find" it, because "these libraries unfortunately they grow in size and entropy," and searching for `tanh` returns "2,800 results" [transcript]. If the person teaching the course gives up on reading the library, a beginner has no chance of learning from it.

## 2.2 What existed before

- **University courses** (Stanford CS231n, which Karpathy taught in 2015–2016): rigorous, organized around vision and matrix-calculus notation. High barrier to entry.
- **Textbooks** (Goodfellow, Bengio, Courville, 2016): comprehensive, mathematically heavy. Excellent reference, punishing first exposure.
- **Top-down practical courses** (fast.ai): results fast, then peel back layers. Effective for many, and it defers the mechanics, which some learners never return for.
- **Visual explainers** (3Blue1Brown): outstanding intuition, no code, stops before transformers.

None does the specific thing this course does: build every piece in front of you, in running code, in order, with nothing hidden, from one derivative to a working GPT.

## 2.3 What breaks without it

Each of these is demonstrated live in a lecture, and each is a failure you will personally reproduce in this book:

- **You cannot tell a broken run from a slow one.** Without knowing a 27-way classifier starts at 3.2958, a run starting at 27.88 looks like "training in progress" rather than "your initialization is wrong." [verified, Chapter 4]
- **You cannot debug dead networks.** In Chapter 4 you will find 61% of a hidden layer saturated flat at initialization, contributing almost no gradient. Nothing errors. The model just trains worse than it should. [verified]
- **You blame the model for the tokenizer's crimes.** Chapter 8's central claim: bad spelling, bad arithmetic, and a measured 7.5× token cost for Hindi versus English all trace to the text-chopping step nobody thinks about. [verified]
- **You cannot make anything fast.** Chapter 9 gets an 11× speedup from understanding hardware, which no framework applies for you. [transcript]

---

# PART III — THE THING ITSELF

*Nine chapters, one per lecture. Each states the problem, builds the code you can run, walks the mechanism step by step, and closes with exercises and a summary you could say out loud.*